# W4 Results Evaluation (molecule, facts_50k)

Evaluate baselines against brute-force ground truth for W4 (multi-signal weighted k-NN WITH predicates). Two signals per query: `fact_text_emb` (L2) + `mol_ecfp` (jaccard), aggregated by `weighted_sum`, then filtered by W2-style predicates.

**Recall**: for each query, count baseline results with `score <= worst_GT_score`. `recall = hits / K`.

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path(".")
GT_FILE = "w4_queries_100_gt.json"

def load_results(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def gt_answer_to_scores(answer):
    return [(row["id"], row["score"]) for row in answer]

def baseline_answer_to_scores(answer):
    return [(row[0], row[-1]) for row in answer]

def query_results_to_map(data, is_gt=False):
    fn = gt_answer_to_scores if is_gt else baseline_answer_to_scores
    return {r["query_id"]: fn(r["answer"]) for r in data["results"]}

assert (RESULTS_DIR / GT_FILE).exists(), f"Ground truth not found: {GT_FILE}"
gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_by_qid = query_results_to_map(gt_data, is_gt=True)

BASELINE_FILES = sorted(
    f.name for f in RESULTS_DIR.glob("*.json")
    if f.name != GT_FILE
)

print(f"Ground truth: {GT_FILE}, queries = {len(gt_by_qid)}")
print(f"Baselines ({len(BASELINE_FILES)}):")
for f in BASELINE_FILES:
    print(f"  {f}")

Ground truth: w4_queries_100_gt.json, queries = 100
Baselines (5):
  20260417_021657.json
  20260417_021705.json
  20260417_021713.json
  20260417_021721.json
  20260417_021729.json


In [2]:
def eval_one_baseline(baseline_by_qid, gt_by_qid):
    """recall = |{baseline rows with score <= worst_GT_score}| / K."""
    per_query = []
    total_hits = 0
    total_denom = 0
    for qid, answers in baseline_by_qid.items():
        if qid not in gt_by_qid:
            continue
        gt_answers = gt_by_qid[qid]
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(score for _, score in gt_answers)
        m = sum(1 for _, score in answers if score <= an + 1e-6)
        recall = m / n
        per_query.append({"query_id": qid, "K": n, "hits": m, "recall": recall})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
for f in BASELINE_FILES:
    data = load_results(RESULTS_DIR / f)
    by_qid = query_results_to_map(data, is_gt=False)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    name = Path(f).stem
    n_queries = len(per_query)
    total_sec = data.get("total_elapsed_sec", None)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        "baseline": name,
        "mean_recall": mean_recall,
        "n_queries": len(per_query),
        "qps": qps,
    })

In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

       baseline  mean_recall  n_queries       qps
20260417_021657        0.991        100 17.718973
20260417_021705        0.991        100 17.689112
20260417_021713        0.991        100 17.782820
20260417_021721        0.991        100 17.603204
20260417_021729        0.991        100 17.696606
